# Praktikum Modul 10 - Dasar-Dasar Deteksi Obyek

Pastikan Anda telah menyesuaikan path dataset yang berada di Google Drive Anda dengan path yang ada di notebook ini.


## D1. Praktikum Deteksi Obyek Dasar

### 1. Import Library dan Akses GDrive


In [ ]:
#import library yang dibutuhkan
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt

#akses drive
from google.colab import drive
drive.mount('/content/drive')


### 2. Implementasi 6 metode template matching
Menggunakan gambar `cats_and_bunnies.jpg` dan `cat2_templatejpg.jpg`.


In [ ]:
# Path disesuaikan dengan folder di drive
img_rgb = cv.imread('/content/drive/MyDrive/1M30vcqgPBD1PHQpqFUtxWdtiMrwSpijC/cats_and_bunnies.jpg')
img_gray = cv.cvtColor(img_rgb, cv.COLOR_BGR2GRAY)
template = cv.imread('/content/drive/MyDrive/1M30vcqgPBD1PHQpqFUtxWdtiMrwSpijC/cat2_templatejpg.jpg', 0)
w, h = template.shape[::-1]

# Menampilkan gambar awal
plt.figure(figsize=(10, 5))
plt.subplot(121), plt.imshow(cv.cvtColor(img_rgb, cv.COLOR_BGR2RGB)), plt.title('input image')
plt.subplot(122), plt.imshow(template, cmap='gray'), plt.title('template image')
plt.show()

methods = ['cv.TM_CCOEFF', 'cv.TM_CCOEFF_NORMED', 'cv.TM_CCORR',
            'cv.TM_CCORR_NORMED', 'cv.TM_SQDIFF', 'cv.TM_SQDIFF_NORMED']

for meth in methods:
    img = img_rgb.copy()
    method = eval(meth)

    # Apply template Matching
    res = cv.matchTemplate(img_gray, template, method)
    min_val, max_val, min_loc, max_loc = cv.minMaxLoc(res)

    if method in [cv.TM_SQDIFF, cv.TM_SQDIFF_NORMED]:
        top_left = min_loc
    else:
        top_left = max_loc
    
    bottom_right = (top_left[0] + w, top_left[1] + h)
    cv.rectangle(img, top_left, bottom_right, (255, 0, 0), 2)

    plt.figure(figsize=(10, 4))
    plt.suptitle(meth, fontsize=16)
    plt.subplot(121), plt.imshow(res, cmap='gray'), plt.title('Matching Result'), plt.xticks([]), plt.yticks([])
    plt.subplot(122), plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB)), plt.title('Detected Point'), plt.xticks([]), plt.yticks([])
    plt.show()


### 3. Template Matching Tanpa Library (Multiple Object)
Menggunakan `bahrain.jpg` dan `bahrain-template.jpg`.


In [ ]:
img_rgb = cv.imread('/content/drive/MyDrive/1M30vcqgPBD1PHQpqFUtxWdtiMrwSpijC/bahrain.jpg')
img_gray = cv.cvtColor(img_rgb, cv.COLOR_BGR2GRAY)
template = cv.imread('/content/drive/MyDrive/1M30vcqgPBD1PHQpqFUtxWdtiMrwSpijC/bahrain-template.jpg', 0)
w, h = template.shape[::-1]

plt.figure(figsize=(10,5))
plt.subplot(121), plt.imshow(cv.cvtColor(img_rgb, cv.COLOR_BGR2RGB)), plt.title('input image')
plt.subplot(122), plt.imshow(template, cmap='gray'), plt.title('template image')
plt.show()

res = cv.matchTemplate(img_gray, template, cv.TM_CCOEFF_NORMED)
threshold = 0.8
loc = np.where(res >= threshold)

for pt in zip(*loc[::-1]):
    cv.rectangle(img_rgb, pt, (pt[0] + w, pt[1] + h), (0,255,255), 2)

plt.figure(figsize=(7,7))
plt.imshow(cv.cvtColor(img_rgb, cv.COLOR_BGR2RGB))
plt.show()


### 4. Edge Detection (Sobel, Canny, Laplacian)
Menggunakan gambar `car-park.jpg`.


In [ ]:
img = cv.imread('/content/drive/MyDrive/1M30vcqgPBD1PHQpqFUtxWdtiMrwSpijC/car-park.jpg', 0)

# Sobel Edge Detection
sobelx = cv.Sobel(img, cv.CV_64F, 1, 0, ksize=3)
sobely = cv.Sobel(img, cv.CV_64F, 0, 1, ksize=3)
sobel_combined = cv.magnitude(sobelx, sobely)

plt.figure(figsize=(12, 4))
plt.subplot(121), plt.imshow(img, cmap='gray'), plt.title('a. Original (Sobel Input)')
plt.subplot(122), plt.imshow(sobel_combined, cmap='gray'), plt.title('Sobel Edge Detection')
plt.show()

# Canny Edge Detection
edges = cv.Canny(img, 100, 200)
plt.figure(figsize=(12, 4))
plt.subplot(121), plt.imshow(img, cmap='gray'), plt.title('b. Original (Canny Input)')
plt.subplot(122), plt.imshow(edges, cmap='gray'), plt.title('Canny Edge Detection')
plt.show()

# Laplacian Edge Detection
# Blur gambar dulu agar lebih baik hasilnya
img_blur = cv.GaussianBlur(img, (3,3), 0)
laplacian = cv.Laplacian(img_blur, cv.CV_64F)
plt.figure(figsize=(12, 4))
plt.subplot(121), plt.imshow(img, cmap='gray'), plt.title('c. Original (Laplacian Input)')
plt.subplot(122), plt.imshow(np.absolute(laplacian), cmap='gray'), plt.title('Laplacian Edge Detection')
plt.show()


### 5. Corner Detection (Harris & Shi-Tomasi)
Menggunakan `chess-board.jpg`.


In [ ]:
img_harris = cv.imread('/content/drive/MyDrive/1M30vcqgPBD1PHQpqFUtxWdtiMrwSpijC/chess-board.jpg')
gray_harris = cv.cvtColor(img_harris, cv.COLOR_BGR2GRAY)
gray_harris = np.float32(gray_harris)
dst = cv.cornerHarris(gray_harris, 2, 3, 0.04)
dst = cv.dilate(dst, None)
img_harris[dst > 0.01 * dst.max()] = [0, 0, 255]

img_shitomasi = cv.imread('/content/drive/MyDrive/1M30vcqgPBD1PHQpqFUtxWdtiMrwSpijC/chess-board.jpg')
gray_shitomasi = cv.cvtColor(img_shitomasi, cv.COLOR_BGR2GRAY)
corners = cv.goodFeaturesToTrack(gray_shitomasi, 100, 0.01, 10)
corners = np.int0(corners)
for i in corners:
    x, y = i.ravel()
    cv.circle(img_shitomasi, (x, y), 5, (0, 0, 255), -1)

plt.figure(figsize=(10, 5))
plt.subplot(121), plt.imshow(cv.cvtColor(img_harris, cv.COLOR_BGR2RGB)), plt.title('a. Harris Corner Detection')
plt.subplot(122), plt.imshow(cv.cvtColor(img_shitomasi, cv.COLOR_BGR2RGB)), plt.title('b. Shi-Tomasi Detection')
plt.show()


### 6. Grid Detection (Hough Transform)
Menggunakan `sudoku.jpg`.


In [ ]:
img = cv.imread('/content/drive/MyDrive/1M30vcqgPBD1PHQpqFUtxWdtiMrwSpijC/sudoku.jpg')
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
edges = cv.Canny(gray, 50, 150, apertureSize=3)
kernel = np.ones((3,3), np.uint8)
edges = cv.dilate(edges, kernel, iterations=1)
edges = cv.erode(edges, kernel, iterations=1)
lines = cv.HoughLines(edges, 1, np.pi/180, 200)

for line in lines:
    rho, theta = line[0]
    a = np.cos(theta)
    b = np.sin(theta)
    x0 = a * rho
    y0 = b * rho
    x1 = int(x0 + 1000 * (-b))
    y1 = int(y0 + 1000 * (a))
    x2 = int(x0 - 1000 * (-b))
    y2 = int(y0 - 1000 * (a))
    cv.line(img, (x1, y1), (x2, y2), (0, 0, 255), 2)

plt.figure(figsize=(8, 8))
plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
plt.title('Hough Transform - Grid Detection')
plt.show()


### 7. Contour Detection
Menggunakan `laptop.jpg`.


In [ ]:
img = cv.imread('/content/drive/MyDrive/1M30vcqgPBD1PHQpqFUtxWdtiMrwSpijC/laptop.jpg')
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
ret, thresh = cv.threshold(img_gray, 127, 255, 0)
contours, hierarchy = cv.findContours(thresh, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)
cv.drawContours(img, contours, -1, (0, 255, 0), 3)

plt.figure(figsize=(10, 6))
plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
plt.title('Contour Detection')
plt.show()


## D2. Praktikum Feature Matching dan Face Detection
### 8 & 9. Face Detection


In [ ]:
cascade_wajah = cv.CascadeClassifier(cv.data.haarcascades + 'haarcascade_frontalface_default.xml')

# List file yang akan diuji
files = ['prabowo.jpg', 'kartini.jpg', 'cat.jpg', 'mask.jpg', 'jordan.jpg', 'solvay.jpg']
path_folder = '/content/drive/MyDrive/1M30vcqgPBD1PHQpqFUtxWdtiMrwSpijC/'

for f in files:
    try:
        img = cv.imread(path_folder + f)
        gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
        
        # Coba deteksi wajah manusia dulu
        faces = cascade_wajah.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
        
        # Jika file adalah cat.jpg dan tidak terdeteksi pakai frontalface, pakai haarcascade khusus kucing
        if f == 'cat.jpg' and len(faces) == 0:
            # Unduh atau pastikan haarcascade_frontalcatface.xml ada
            cascade_cat = cv.CascadeClassifier(cv.data.haarcascades + 'haarcascade_frontalcatface.xml')
            faces = cascade_cat.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
            
        for (x, y, w, h) in faces:
            cv.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
            
        plt.figure(figsize=(5, 5))
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        plt.title(f'Deteksi Wajah pada {f}')
        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"Gagal memproses file {f}: {e}")


### 10. Eyes Detection (Prabowo)


In [ ]:
cascade_mata = cv.CascadeClassifier(cv.data.haarcascades + 'haarcascade_eye.xml')
img = cv.imread(path_folder + 'prabowo.jpg')
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
faces = cascade_wajah.detectMultiScale(gray, 1.3, 5)

for (x, y, w, h) in faces:
    roi_gray = gray[y:y+h, x:x+w]
    roi_color = img[y:y+h, x:x+w]
    eyes = cascade_mata.detectMultiScale(roi_gray)
    for (ex, ey, ew, eh) in eyes:
        cv.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (255, 0, 0), 2)

plt.figure(figsize=(5, 5))
plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
plt.title('Eyes Detection')
plt.axis('off')
plt.show()


### 11. Smile Detection (People)


In [ ]:
cascade_smile = cv.CascadeClassifier(cv.data.haarcascades + 'haarcascade_smile.xml')
img = cv.imread(path_folder + 'people.jpg') # Asumsi nama file
if img is not None:
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    faces = cascade_wajah.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        roi_color = img[y:y+h, x:x+w]
        smiles = cascade_smile.detectMultiScale(roi_gray, 1.8, 20)
        for (sx, sy, sw, sh) in smiles:
            cv.rectangle(roi_color, (sx, sy), (sx+sw, sy+sh), (0, 0, 255), 2)

    plt.figure(figsize=(5, 5))
    plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    plt.title('Smile Detection')
    plt.axis('off')
    plt.show()
else:
    print("File gambar people.jpg tidak ditemukan.")


### 13. Blurring pada wajah (Jordan)


In [ ]:
img = cv.imread(path_folder + 'jordan.jpg')
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
faces = cascade_wajah.detectMultiScale(gray, 1.3, 5)

for (x, y, w, h) in faces:
    # ROI (Region of Interest) dari wajah yang dideteksi
    roi = img[y:y+h, x:x+w]
    # Lakukan blurring pada ROI wajah
    roi = cv.medianBlur(roi, 35)
    # Ganti bagian wajah pada gambar dengan wajah yang sudah diblur
    img[y:y+h, x:x+w] = roi

plt.figure(figsize=(5, 5))
plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
plt.title('Blurred Face')
plt.axis('off')
plt.show()


### 14. Mengolah gambar KTM
Anda perlu menyiapkan file foto KTM (`ktm.jpg`) dan gambar keseluruhan berisi KTM (`ktm_barang.jpg`) pada folder dataset Anda di Drive.


In [ ]:
# a & b. Tampilkan Foto KTM
try:
    ktm = cv.imread(path_folder + 'ktm.jpg')
    plt.figure(figsize=(6, 4))
    plt.imshow(cv.cvtColor(ktm, cv.COLOR_BGR2RGB))
    plt.title('FOTO KTM')
    plt.axis('off')
    plt.show()
    
    # c. Deteksi obyek KTM menggunakan Feature Matching (SIFT)
    img_barang = cv.imread(path_folder + 'ktm_barang.jpg')
    sift = cv.SIFT_create()
    
    kp1, des1 = sift.detectAndCompute(ktm, None)
    kp2, des2 = sift.detectAndCompute(img_barang, None)
    
    bf = cv.BFMatcher()
    matches = bf.knnMatch(des1, des2, k=2)
    
    good = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good.append([m])
            
    sift_matches = cv.drawMatchesKnn(ktm, kp1, img_barang, kp2, good, None, flags=2)
    
    plt.figure(figsize=(15, 8))
    plt.imshow(cv.cvtColor(sift_matches, cv.COLOR_BGR2RGB))
    plt.title('Feature Matching KTM (SIFT)')
    plt.axis('off')
    plt.show()
    
except Exception as e:
    print("Harap siapkan 'ktm.jpg' dan 'ktm_barang.jpg' di Google Drive Anda.\n", e)
